# Backend API Test Notebook
Use this notebook to smoke-test FastAPI endpoints.

Set `API_BASE` in the next cell to either your Render URL or `http://localhost:3000` for local dev.

> **Note:** The Render free tier spins down after 15 minutes of inactivity. The first request after a cold start can take 30–60 seconds — just wait and retry.


In [1]:
import json
from typing import Any
import requests

LOCAL_URL = "http://localhost:3000"
RENDER_URL = "https://london-explorer.onrender.com"  # ← paste your Render URL here
TIMEOUT_SECONDS = 30

# Auto-select: use local if it's up, otherwise fall back to Render
try:
    requests.get(f"{LOCAL_URL}/health", timeout=2)
    API_BASE = LOCAL_URL
    print(f"✓ Local server detected — using {API_BASE}")
except requests.exceptions.ConnectionError:
    API_BASE = RENDER_URL
    print(f"✓ Local server not running — using {API_BASE}")


✓ Local server not running — using https://london-explorer.onrender.com


In [2]:


def call_api(path: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{API_BASE}{path}"
    response = requests.get(url, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json()

def preview(payload: Any, max_items: int = 3):
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        data = payload["data"]
        summary = {k: v for k, v in payload.items() if k != "data"}
        print("Summary:")
        print(json.dumps(summary, indent=2))
        print("\nData preview:")
        print(json.dumps(data[:max_items], indent=2))
        print(f"\nData length: {len(data)}")
        return

    print(json.dumps(payload, indent=2))


## 1) Health Check

In [3]:
health = call_api("/health")
preview(health)

{
  "status": "ok"
}


## 2) Tiles Endpoint
Adjust the viewport/filter params as needed.

In [4]:
tiles_params = {
    "sw_lat": 51.48,
    "sw_lng": -0.22,
    "ne_lat": 51.54,
    "ne_lng": -0.06,
    "zoom": 13,
    "cuisine": "",
    "cost": "",
    "venue_type": "",
    "score_basis": 0,
    "confidence": 1,
    "score_tier": 0,
}

tiles = call_api("/api/tiles", params=tiles_params)
preview(tiles)

Summary:
{
  "mode": "tiles",
  "resolution": 8
}

Data preview:
[
  {
    "tile": "88194ad00dfffff",
    "count": 13
  },
  {
    "tile": "88194ad029fffff",
    "count": 21
  },
  {
    "tile": "88194ad067fffff",
    "count": 22
  }
]

Data length: 212


## 3) Nearby Endpoint

In [5]:
nearby_params = {
    "lat": 51.5074,
    "lng": -0.1278,
    "radius_m": 1000,
    "cuisine": "",
    "cost": "",
    "venue_type": "",
    "score_basis": 0,
    "confidence": 1,
    "rank_threshold": 0,
    "page": 1,
}

nearby = call_api("/api/nearby", params=nearby_params)
preview(nearby)

Summary:
{
  "page": 1,
  "page_size": 20
}

Data preview:
[
  {
    "id": "ChIJV8gP0ykFdkgRbsmzJFpywNA",
    "display_name": "The Ritz Restaurant",
    "lat": 51.5069213,
    "lon": -0.1419736999999999,
    "cuisine_type": "Fine Dining",
    "venue_type": "Dine-In",
    "cost": "100+",
    "rating": 4.699999809265137,
    "user_rating_count": 1385,
    "operational": true,
    "rank": 0.9998472332954407
  },
  {
    "id": "ChIJy23uOwAFdkgRii6cDLJ0Z8k",
    "display_name": "YiQi Pan Asia",
    "lat": 51.5115094,
    "lon": -0.1308137,
    "cuisine_type": "Chinese",
    "venue_type": "Dine-In",
    "cost": "20+",
    "rating": 4.900000095367432,
    "user_rating_count": 3948,
    "operational": true,
    "rank": 0.9986250996589661
  },
  {
    "id": "ChIJu8QEs-8FdkgRE0znNGEGBoQ",
    "display_name": "Row on 5",
    "lat": 51.510497,
    "lon": -0.1398547,
    "cuisine_type": "Fine Dining",
    "venue_type": "Dine-In",
    "cost": "100+",
    "rating": 4.900000095367432,
    "user_rating

## 4) Place Detail Endpoint
Run the next cell after running nearby/tiles so you can pick a real place id.

In [6]:
sample_place_id = nearby.get("data", [{}])[0].get("id") if isinstance(nearby, dict) else None
sample_place_id

'ChIJV8gP0ykFdkgRbsmzJFpywNA'

In [ ]:
if not sample_place_id:
    raise ValueError("No place id available. Set sample_place_id manually and retry.")

place = call_api(f"/api/place/{sample_place_id}")
preview(place)